# 图片爬取部分

## Wikipedia Commons

In [ ]:
import httpx
from bs4 import BeautifulSoup
import re
import json
import pandas as pd
import os
import random
import time
current_dir = os.getcwd()
PROJECT_ROOT = os.path.dirname(os.path.dirname(current_dir))
PROJECT_ROOT

###  Wikipedia commons标签处理

由于commons的标签全都由英文书写，而且对于包含片假名形式格式是按照罗马音处理的，但是又有些格式不统一，为了保证能正确搜索到Category，需要进行转换

In [ ]:
OPERATOR_PREFIX = {
    "JR東日本": "JR East",
    "JR東海":   "JR Central",
    "JR西日本": "JR West",
    "JR九州":   "JR Kyushu",
    "JR北海道": "JR Hokkaido",
    "JR四国":   "JR Shikoku",
    "JR貨物":   "JR Freight",
}

_DIGRAPHS = {
    'キャ':'kya','キュ':'kyu','キョ':'kyo',
    'シャ':'sha','シュ':'shu','ショ':'sho',
    'チャ':'cha','チュ':'chu','チョ':'cho',
    'ニャ':'nya','ニュ':'nyu','ニョ':'nyo',
    'ヒャ':'hya','ヒュ':'hyu','ヒョ':'hyo',
    'ミャ':'mya','ミュ':'myu','ミョ':'myo',
    'リャ':'rya','リュ':'ryu','リョ':'ryo',
    'ギャ':'gya','ギュ':'gyu','ギョ':'gyo',
    'ジャ':'ja', 'ジュ':'ju', 'ジョ':'jo',
    'ビャ':'bya','ビュ':'byu','ビョ':'byo',
    'ピャ':'pya','ピュ':'pyu','ピョ':'pyo',
}
_SINGLE = {
    'ア':'a', 'イ':'i', 'ウ':'u', 'エ':'e', 'オ':'o',
    'カ':'ka','キ':'ki','ク':'ku','ケ':'ke','コ':'ko',
    'サ':'sa','シ':'shi','ス':'su','セ':'se','ソ':'so',
    'タ':'ta','チ':'chi','ツ':'tsu','テ':'te','ト':'to',
    'ナ':'na','ニ':'ni','ヌ':'nu','ネ':'ne','ノ':'no',
    'ハ':'ha','ヒ':'hi','フ':'fu','ヘ':'he','ホ':'ho',
    'マ':'ma','ミ':'mi','ム':'mu','メ':'me','モ':'mo',
    'ヤ':'ya','ユ':'yu','ヨ':'yo',
    'ラ':'ra','リ':'ri','ル':'ru','レ':'re','ロ':'ro',
    'ワ':'wa','ヲ':'wo','ン':'n',
    'ガ':'ga','ギ':'gi','グ':'gu','ゲ':'ge','ゴ':'go',
    'ザ':'za','ジ':'ji','ズ':'zu','ゼ':'ze','ゾ':'zo',
    'ダ':'da','デ':'de','ド':'do',
    'バ':'ba','ビ':'bi','ブ':'bu','ベ':'be','ボ':'bo',
    'パ':'pa','ピ':'pi','プ':'pu','ペ':'pe','ポ':'po',
}

# Commons 分类特例：key=(series, wiki_title 去掉 #anchor)，value=直接使用的搜索前缀
COMMONS_OVERRIDES: dict[tuple[str, str], str] = {
    ("481系", "国鉄485系電車"): "JNR 485",  # 481/483/485/489系同属485系 category
}


def parse_json_list(value: str) -> list[str]:
    return json.loads(value)


def _katakana_to_romaji(text: str) -> str:
    result, i = '', 0
    while i < len(text):
        two = text[i:i+2]
        if two in _DIGRAPHS:
            result += _DIGRAPHS[two]; i += 2
        elif text[i] in _SINGLE:
            result += _SINGLE[text[i]]; i += 1
        else:
            result += text[i]; i += 1
    return result


def _operator_prefixes(operator_jp: list[str], series_type: str, wiki_title: str) -> list[str]:
    if series_type == "新幹線電車":
        return ["Shinkansen"]
    if wiki_title.startswith("国鉄"):
        return ["JNR"]

    prefixes = []
    for op in operator_jp:
        prefix = OPERATOR_PREFIX.get(op, op)
        if prefix not in prefixes:
            prefixes.append(prefix)
    return prefixes


def series_to_commons_prefixes(series: str, operator_jp: list[str], series_type: str, wiki_title: str) -> list[str]:
    wiki_base = wiki_title.split("#", 1)[0]
    override = COMMONS_OVERRIDES.get((series, wiki_base))
    if override:
        return [override]

    name = re.sub(r'[系形]$', '', series)
    m = re.match(r'^([゠-ヿ]+)(.*)', name)

    prefixes = []
    for op in _operator_prefixes(operator_jp, series_type, wiki_title):
        if m:
            romaji = _katakana_to_romaji(m.group(1)).capitalize()
            rest = m.group(2)
            candidates = [f"{op} {romaji} {rest}".strip()]
            if rest:
                candidates.append(f"{op} {rest}")
        else:
            candidates = [f"{op} {name}"]

        for prefix in candidates:
            if prefix not in prefixes:
                prefixes.append(prefix)

    return prefixes


def fetch_commons_categories(prefix: str, max_retries: int = 3, base_sleep: float = 1.0) -> list[str] | None:
    for attempt in range(max_retries + 1):
        try:
            resp = httpx.get(
                "https://commons.wikimedia.org/w/api.php",
                params={
                    "action":   "query",
                    "list":     "allcategories",
                    "acprefix": prefix,
                    "aclimit":  5,
                    "format":   "json",
                },
                headers={"User-Agent": "TrainDatasetBuilder/1.0 (research; fengyukunfyk@gmail.com)"},
                timeout=30,
            )
            resp.raise_for_status()
            data = resp.json()

            if "query" in data:
                return [r["*"] for r in data["query"].get("allcategories", [])]

            error = data.get("error", {})
            code = error.get("code", "missing-query")
            info = error.get("info", data)
            raise RuntimeError(f"Commons API error for {prefix}: {code} {info}")

        except (httpx.HTTPError, ValueError, RuntimeError) as exc:
            if attempt >= max_retries:
                print(f'⚠ {prefix}: request failed after {max_retries + 1} attempts: {exc}')
                return None

            sleep_s = base_sleep * (2 ** attempt) + random.uniform(0, 0.5)
            print(f'⚠ {prefix}: temporary request error, retrying in {sleep_s:.1f}s ({attempt + 1}/{max_retries})')
            time.sleep(sleep_s)


def search_commons_category(
    series: str, operator_jp: list[str], series_type: str, wiki_title: str
) -> tuple[str, list[str]]:
    prefixes = series_to_commons_prefixes(series, operator_jp, series_type, wiki_title)
    for prefix in prefixes:
        cats = fetch_commons_categories(prefix)
        if cats is None:
            continue
        if cats:
            return prefix, cats
    return prefixes[0], []


all_model = pd.read_csv(os.path.join(PROJECT_ROOT, "data", "jr_honshu_series.csv"))
for col in ["operator_page_title", "operator_jp", "operator_en"]:
    all_model[col] = all_model[col].apply(parse_json_list)


In [ ]:
commons_prefixes, commons_cats = [], []

for _, row in all_model.iterrows():
    prefix, cats = search_commons_category(
        row["series"], row["operator_jp"], row["type"], row["wiki_title"]
    )
    commons_prefixes.append(prefix)
    commons_cats.append(cats)
    status = "✓" if cats else "✗"
    print(f'{status} {row["series"]} ({", ".join(row["operator_jp"])}) → "{prefix}"  {cats}')

all_model["commons_prefix"] = commons_prefixes
all_model["commons_cats"] = commons_cats

print(f'\n共 {len(all_model)} 条，命中 {all_model["commons_cats"].apply(bool).sum()} 条，'
      f'空结果 {(~all_model["commons_cats"].apply(bool)).sum()} 条')
all_model[["series", "operator_jp", "commons_prefix", "commons_cats"]]
